# 🔗 Notebook 8: Full Medical Book RAG + LLM Pipeline
**Goal:** Connect Hybrid Book Retriever → Qwen 2.5 2B (LoRA fine-tuned) into a complete end-to-end system.

**Pipeline:** User Query → Search Textbooks via Vector/KG → Compile Context → LLM → Safe Gujarati Answer

**Output:** Working end-to-end system, `pipeline.py` module for Streamlit

In [ ]:
import os, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from retriever import hybrid_retrieve

print('✅ Imports done')

## 1. Load Fine-tuned Qwen 2.5 2B (Base + LoRA Adapter)

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct' # Using user updated 3B base if available, standardly 2B
ADAPTER_PATH = 'models/qwen_gu_health_lora'
MY_TOKEN = ""

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
) if torch.cuda.is_available() else None

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH if os.path.exists(ADAPTER_PATH) else BASE_MODEL,
    trust_remote_code=True,
    token=MY_TOKEN
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto' if torch.cuda.is_available() else 'cpu',
    trust_remote_code=True,
    token=MY_TOKEN
)

# Apply LoRA adapter if available
if os.path.exists(ADAPTER_PATH) and os.path.exists(os.path.join(ADAPTER_PATH, 'adapter_config.json')):
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    print(f'✅ Fine-tuned model loaded (base + LoRA adapter from {ADAPTER_PATH})')
else:
    model = base_model
    print(f'⚠️  LoRA adapter not found, using base model. Run Notebook 05 first.')

model.eval()
print('✅ Model ready for inference')

## 2. System Prompt & Safety Layer (Medical Books aligned)

In [ ]:
SYSTEM_PROMPT = """You are a highly safe, accurate Gujarati healthcare assistant.
Rules you MUST follow:
1. Answer ONLY in Gujarati language, translating any English medical book extracts cleanly to Gujarati.
2. Base your answer SOLELY on the extracted Medical Books context (provided below). Do NOT hallucinate or guess treatments not explicitly mentioned in the medical books.
3. Always append a safety advisory sentence at the end of your response.
4. If you are unsure or the context is empty, say "ડૉક્ટરની સલાહ લો" (consult a doctor).
5. Never recommend dangerous doses, treatments, or self-surgery.
"""

EMERGENCY_KEYWORDS = [
    'heart attack', 'stroke', 'unconscious', 'bleeding', 'poisoning', 'overdose',
    'breathing', 'chest pain', 'collapse', 'ઇમર્જન્સી', 'અવ્યયવ', 'ઝેર', 'બેભાન',
    'severe', 'ગંભીર', 'emergency',
]

EMERGENCY_RESPONSE = (
    "⚠️ **તાત્કાલિક:** \n\n"
    "આ ગંભીર સ્વાસ્થ્ય સ્થિતિ છે. **108 (Ambulance)** અથવા "
    "**112 (Emergency)** ને અત્યારે જ call કરો. "
    "કૃપા કરીને ઘરેલૂ ઉપચાર ન કરો."
)

def is_emergency(query: str) -> bool:
    q = query.lower()
    return any(kw.lower() in q for kw in EMERGENCY_KEYWORDS)

print('✅ System prompt and safety layer configured')

## 3. RAG Answer Generation Function

In [ ]:
def rag_answer(query: str, top_k: int = 5, max_new_tokens: int = 350) -> dict:
    """
    Full RAG pipeline:
    1. Emergency check
    2. Hybrid retrieval (Medical Book Vectors + book KG)
    3. LLM answer translation/generation with context
    """
    if is_emergency(query):
        return {
            'query': query,
            'answer': EMERGENCY_RESPONSE,
            'is_emergency': True,
            'context': '',
            'kg_results': {}
        }

    retrieval = hybrid_retrieve(query, top_k=top_k)
    context = retrieval['combined_context']

    user_message = f"""Medical Textbook Context:
{context}

Query (in Gujarati): {query}

Please answer the query above in Gujarati using ONLY the context provided."""

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_message}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return {
        'query': query,
        'answer': answer,
        'is_emergency': False,
        'context': context,
        'kg_results': retrieval['kg_results'],
        'vector_results': retrieval['vector_results']
    }

## 4. Run 10+ End-to-End Tests

In [ ]:
test_queries = [
    'ડાયાબિટીઝ ના મુખ્ય લક્ષણો ક્યા છે?',
    'ઉચ્ચ blood pressure ઘટાડવા ઘરેલૂ ઉપાય',
    'Paracetamol ક્યારે અને કેટલી માત્રામાં લઈ શકાય?',
    'ઘૂંટણ ના દુખાવા ની સારવાર',
    'dengue fever symptoms',
    'ડૉક્ટર ને ક્યારે મળવું જોઈએ?',
    'cholesterol ઘટાડવા ખોરાક',
    'malaria prevention',
    'heart attack symptoms emergency',     # Emergency test
    'ઊંઘ ન આવવી - Insomnia ઉપાય',
]

results_log = []

print('🧪 End-to-End RAG Pipeline Test (Using Medical Books)')
print('='*70)

for q in test_queries:
    result = rag_answer(q)
    print(f'\nQ: {q}')
    if result['is_emergency']:
        print('⚠️  EMERGENCY RESPONSE triggered')
    print(f'A: {result["answer"][:300]}...' if len(result["answer"]) > 300 else f'A: {result["answer"]}')
    print(f'KG: entities={result["kg_results"].get("matched_entities", [])[:3]}')
    print('-'*50)
    results_log.append(result)

os.makedirs('outputs', exist_ok=True)
with open('outputs/pipeline_test_results.json', 'w', encoding='utf-8') as f:
    serializable = [{k: v for k, v in r.items() if k != 'vector_results'} for r in results_log]
    json.dump(serializable, f, ensure_ascii=False, indent=2)

print('\n✅ Pipeline test complete. Results saved to outputs/pipeline_test_results.json')

## 5. Save Pipeline Module

In [ ]:
pipeline_code = '''
"""pipeline.py — Full RAG pipeline for Gujarati Healthcare QA (Books Based)."""
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from retriever import hybrid_retrieve

BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH = "models/qwen_gu_health_lora"
MY_TOKEN = ""

SYSTEM_PROMPT = (
    "You are a safe, accurate Gujarati healthcare assistant. "
    "Answer ONLY in Gujarati. Translate English textbook notes cleanly to Gujarati natively. "
    "Use ONLY the provided context from medical books. "
    "Do NOT hallucinate. Always end with a safety advisory."
)

EMERGENCY_KEYWORDS = [
    "heart attack", "stroke", "unconscious", "bleeding", "poisoning",
    "overdose", "chest pain", "collapse", "ઇમર્જન્સી", "ઝેર", "ગંભીર"
]

EMERGENCY_RESPONSE = (
    "⚠️ આ ગંભીર સ્વાસ્થ્ય સ્થિતિ છે. **108 (Ambulance)** "
    "અથવા **112** ને અત્યારે જ call કરો. "
    "ઘરેલૂ ઉપચાર ન કરો."
)

def _load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16) if torch.cuda.is_available() else None
    tok_path = ADAPTER_PATH if os.path.exists(ADAPTER_PATH) else BASE_MODEL
    tok = AutoTokenizer.from_pretrained(tok_path, trust_remote_code=True, token=MY_TOKEN)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb,
        device_map="auto" if torch.cuda.is_available() else "cpu", trust_remote_code=True, token=MY_TOKEN)
    if os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")):
        mdl = PeftModel.from_pretrained(base, ADAPTER_PATH)
    else:
        mdl = base
    mdl.eval()
    return tok, mdl

_tokenizer, _model = _load_model()

def is_emergency(query):
    q = query.lower()
    return any(kw.lower() in q for kw in EMERGENCY_KEYWORDS)

def answer(query, top_k=5, max_new_tokens=350):
    if is_emergency(query):
        return {"query": query, "answer": EMERGENCY_RESPONSE, "is_emergency": True, "context": "", "kg_results": {}}
    retr = hybrid_retrieve(query, top_k=top_k)
    ctx  = retr["combined_context"]
    msg  = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Medical Textbook Context:\n{ctx}\n\nQuery: {query}"}]
    txt  = _tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inp  = _tokenizer(txt, return_tensors="pt").to(_model.device)
    with torch.no_grad():
        out = _model.generate(**inp, max_new_tokens=max_new_tokens, temperature=0.6,
                               do_sample=True, top_p=0.9, repetition_penalty=1.1,
                               pad_token_id=_tokenizer.eos_token_id)
    ans = _tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return {"query": query, "answer": ans, "is_emergency": False,
            "context": ctx, "kg_results": retr["kg_results"],
            "vector_results": retr["vector_results"]}
'''

with open('pipeline.py', 'w', encoding='utf-8') as f:
    f.write(pipeline_code.strip())

print('✅ pipeline.py saved')
print('\n📝 NEXT STEP: Run Notebook 09 for evaluation.')